In [ ]:
RANDOM_STATE = 2025
import os
import sys
# Get the absolute path of the current directory
current_dir = os.getcwd()

# Get the parent directory (project root)
parent_dir = os.path.dirname(current_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

# Force changes in the local python files to be re-loaded (instead of using the cached version)
%load_ext autoreload
%autoreload 2

In [ ]:

import pandas as pd
import numpy as np
import gc

from data.dataset import MalwareDatasetLoader
from data.data_processing import split_out_targets, force_dense, preprocess_existing, preprocess_fit

RELOAD_DATA = True
if not RELOAD_DATA:
  try:
    print(df_features_train.head())
  except Exception as e:
    print("No dataframe.  Loading data...")
    RELOAD_DATA=True
if RELOAD_DATA:
  df_loader = MalwareDatasetLoader()

  df_train, df_val, df_test = df_loader.make_data_splits()
  
  df_features_train, df_y_train = split_out_targets(df_train)
  df_features_val, df_y_val = split_out_targets(df_val)
  df_features_test, df_y_test = split_out_targets(df_test)
  del df_loader
  gc.collect()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


def compute_metrics(classifier, df_features, df_y):
  print(f"Compute Metrics Start: {df_features.shape[0]}")
  predictions = classifier.predict(df_features)
  print("Compute Metrics End")

  acc = accuracy_score(df_y, predictions)
  f1 = f1_score(df_y, predictions)
  cm = confusion_matrix(df_y, predictions)

  print(f"Accuracy: {acc:.4f}")
  print(f"F1:       {f1:.4f}")
  #print(f"AUC:      {auc:.4f}")
  print("Confusion matrix:")
  print(cm)
  print()
  return classification_report(df_y, predictions, output_dict=True)


In [ ]:

from numpy.random.mtrand import f
import sklearn
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import MiniBatchKMeans
import numpy as np
from sklearn.metrics import classification_report

VERBOSE = False

def transform_kmeans(features, labels, kmeans_clusters_per_class):
    features_train_svm = []
    y_train_svm = []

    unique_labels = np.unique(labels["label"])
    print(unique_labels)
    for label in unique_labels:
        # Filter data for this specific class
        class_data = features[(labels['label'] == label).values]

        kmeans = MiniBatchKMeans(
            n_clusters=kmeans_clusters_per_class, 
            batch_size=4096,
            random_state=42,
            n_init="auto",
        )
    
        kmeans.fit(class_data)
        centers = kmeans.cluster_centers_

        # The cluster centers become our new "training examples"
        features_train_svm.append(centers)
        y_train_svm.append(np.full(centers.shape[0], label))

    features_final = np.vstack(features_train_svm)
    labels_final = np.concatenate(y_train_svm)
    return features_final, labels_final

class SvmClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self):
        super().__init__()
        self.classifier = None
    
    def predict(self, X):
        dense_X_transformed = force_dense(X)
        return self.classifier.predict(dense_X_transformed)

    def train(self, X_final, y_final, C, gamma, kernel):
        print(X_final.shape)

        self.classifier = SVC(
            kernel=kernel,
            C=C,
            gamma=gamma,
           #max_iter=20_000_000,
        )
        
        self.classifier.fit(X_final, y_final)
        print(f"Iterations: {self.classifier.n_iter_}")



In [ ]:
from typing import Optional
from data import dataset
from sklearn.preprocessing import PowerTransformer
from sklearn.base import TransformerMixin, OneToOneFeatureMixin
from data.data_processing import preprocess_fit

def p999(x):
    return x.quantile(0.999)
import gc

ANALYZE_DISTRIBUTIONS = False
if ANALYZE_DISTRIBUTIONS:
    numeric_features_df = df_features_train[dataset.NUMERIC_COLUMNS]
    gc.collect()
    result = numeric_features_df.agg(['min', 'max', p999])
    print(result)
    gc.collect()

    Xprime, preprocessor = preprocess_fit(numeric_features_df, include_onehot=False, quantile_clipping=False, power_transform=True)
    gc.collect()
    new_columns = preprocessor.get_feature_names_out()
    gc.collect()

    # Create a temporary DataFrame for analysis
    df_post = pd.DataFrame(Xprime)#, columns=new_columns)
    result2 = df_post.agg(['min', 'max', 'mean', 'std', p999])
    gc.collect()
    print(result2)


In [ ]:
def run_single(gamma, c, kernel, X_kmeans, y_kmeans, X_validation_transformed):
    print(f"Experiment: Kernel: {kernel} - C: {c} - Gamma: {gamma}")
    classifier = SvmClassifier()
    classifier.train(X_kmeans, y_kmeans, c, gamma, kernel)
    print(f"Support vectors: {classifier.classifier.n_support_}")
    print(f"Percentage of training set: {np.sum(classifier.classifier.n_support_) / len(X_kmeans)}")

    metrics = compute_metrics(classifier.classifier, X_validation_transformed, df_y_val)
    description = f"SVM: - {kernel} - {c} - {gamma}"
    print(description)
    print(f"Class 0: {metrics['0']['f1-score']}  - Class 1:  {metrics['1']['f1-score']}")
    
    return(classifier, metrics, (kernel, c, gamma), description)


In [ ]:

ALL_SVM_KERNEL_TYPES = ['poly', 'linear', 'rbf', 'sigmoid']

# By default this will keep existing results when starting a new run.
# Set this to true to remove old training results.
RESET_RESULTS_LIST=False
try:
  test = final_results[0]
except Exception as e:
  RESET_RESULTS_LIST = True
if RESET_RESULTS_LIST:
  final_results = []

# 3-stage training process:
# 1) Set SINGLE_TRAIN and FINETUNE_TRAIN to false.
#    Train on a wide range of hyper parameters.
# 2) Select the best model, and put the neighborhood where
#    it performed well into the FINETUNE_TRAIN section.
#    This section uses a larger K-Means clustering space.
#    Set FINETUNE_TRAIN = True
# 3) Select the single best config, set SINGLE_TRAIN = True,
#    and run one final time.  (with 30k clusters)
SINGLE_TRAIN = True
FINETUNE_TRAIN = False
REPEAT_RANDOM_STATES = [RANDOM_STATE]
if SINGLE_TRAIN:
  REPEAT_RANDOM_STATES = [RANDOM_STATE]#, 94126, 3321, 2025, 91] + [i for i in range(15)]
  C_values = [100000]
  gamma_values = [0.31622776601683794] 
  kernels=['rbf'] 
  clusters_per_class=10000
elif FINETUNE_TRAIN:
  C_values = np.logspace(start=3, stop=4, num=3)
  gamma_values = np.logspace(start=-1, stop=0, num=9)
  kernels = ALL_SVM_KERNEL_TYPES
  clusters_per_class = 10000
else:
  C_values = np.logspace(start=-4, stop=8, num=9)
  gamma_values = np.logspace(start=-6, stop=5, num=9)
  kernels = ALL_SVM_KERNEL_TYPES
  clusters_per_class = 1000

# There are two sampling methods.  Choose one.
KMEANS_SAMPLING=True
RANDOM_SAMPLING=False
import concurrent
NUM_WORKERS=32
PARALLEL=True
futures = []
total_experimets = 0
with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
  for random_state_seed in REPEAT_RANDOM_STATES:
    if KMEANS_SAMPLING:
      print("Starting KMeans")
      X_transformed, preprocessor = preprocess_fit(df_features_train, quantile_clipping=True, feature_scaling=False, sequential_numeric=True)
      X_final, y_final = transform_kmeans(X_transformed, df_y_train, clusters_per_class)
      print("Finished KMeans")
    elif RANDOM_SAMPLING:
      df_features_train_sample = df_features_train.sample(30000, random_state=random_state_seed)
      y_final = df_y_train.loc[df_features_train_sample.index]
      X_final, preprocessor = preprocess_fit(df_features_train_sample, quantile_clipping=True, feature_scaling=False, sequential_numeric=True)
    else:
      assert False
    X_validation_transformed = force_dense(preprocess_existing(df_features_val, preprocessor))
    for kernel in kernels:
      for c in C_values:
        for gamma in gamma_values:
          if PARALLEL:
            print(f"SCHEDULING: SVM: - {kernel} - {c} - {gamma}")
            total_experimets += 1
            futures.append(executor.submit(run_single,
                                          gamma, c, kernel, X_final, y_final, X_validation_transformed))
          else:
            final_results.append(run_single(gamma, c, kernel, X_final, y_final, X_validation_transformed))

  print(f"Starting to run experiments: COUNT: {total_experimets}")

  for future in concurrent.futures.as_completed(futures):
    result = future.result()
    if result is not None:
      final_results.append(result)



In [ ]:

max_avg_f1 = 0
best_config = None
for (classifier, metrics, (kernel, c, gamma), description) in final_results[10:]:
    avg_f1 = (metrics['0']['f1-score'] + metrics['1']['f1-score'])/2
    if avg_f1 > max_avg_f1:
        max_avg_f1 = avg_f1
        max_f1_preprocessor = preprocessor
        max_f1_classifier = classifier
        best_config = (kernel, c, gamma)

print(f"Best Configuration: {best_config}")
X_test_transformed = preprocess_existing(df_features_test, max_f1_preprocessor)

metrics = compute_metrics(max_f1_classifier, X_test_transformed, df_y_test)


In [ ]:
import matplotlib.pyplot as plt
import math
def graph_results(results, kernel_filters=None):
  fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
  
  # Flatten the 2x2 matrix of axes into a 1D list for easy iteration
  axes_flat = axes.flatten()
  assert kernel_filters is not None
  for index, kernel_filter in enumerate(kernel_filters):
    print(f"Searching for: {kernel_filter}")
    ax = axes_flat[index]

    x_dim_points = []
    y_dim_points = []
    labels = []
    max_f1 = 0
    max_f1_desc = ""
    # FIXME:
    for (classifier, metrics, (kernel, c, gamma), description) in results:
      if kernel != kernel_filter:
        continue
      x_dim_points.append(math.log10(c))
      y_dim_points.append(math.log10(gamma))

      avg_f1 = (metrics['0']['f1-score'] + metrics['1']['f1-score'])/2
      if avg_f1 > max_f1:
        max_f1 = avg_f1
        max_f1_desc = description
      labels.append(avg_f1)
    print(f"Kernel type: {kernel_filter} - max F1: {max_f1} - desc: {max_f1_desc}")
    plot_data = sorted(zip(x_dim_points, y_dim_points, labels), key=lambda k: k[2])
    x_sorted, y_sorted, labels_sorted = zip(*plot_data)
    sc = ax.scatter(x_sorted, y_sorted, c=labels_sorted, cmap='coolwarm', s=30, vmin=0.9, vmax=0.999)
    ax.set_title(f"Kernel: {kernel_filter}")
    ax.set_xlabel("Log10(C)")
    ax.set_ylabel("Log10(Gamma)")
    cbar = plt.colorbar(sc)
    cbar.set_label('Intensity Level') # Optional label
  plt.tight_layout()
  plt.show()
  plt.show()

graph_results(final_results, kernel_filters=ALL_SVM_KERNEL_TYPES)

In [ ]:
last_kernel = ""
for index, (classifier, metrics, (kernel, c, gamma), description) in enumerate(final_results):
    if kernel != last_kernel:
        print(index)
        print(kernel)
    last_kernel = kernel